# 🌍📰 GeoBrief · CrewAI + Ollama · Noticias ONU (RSS) · Versión alumno
**Objetivo:** crear un boletín de geopolítica con tres agentes coordinados y un LLM local.

En esta versión las celdas de programación no contienen la solución. Encontrarás comentarios con **TODO**, objetivos y **pistas** para completar cada parte.

**Aprenderás:** `LLM`, `Agent`, `Task`, herramientas, `context` y `Process.sequential`.

**Resultado esperado:** un dossier, un análisis y un boletín de hasta 200 palabras con referencias a una pequeña base de conocimiento construida a partir del RSS de Noticias ONU.

**Duración orientativa:** 90–120 minutos, con modelo y dependencias preparados antes. Se recomienda Python básico y ejecutar el notebook en Jupyter local o VS Code.

> Importante: intenta resolver primero cada bloque usando las pistas. Consulta la documentación enlazada solo cuando lo necesites.


## 1) Instalar dependencias (una vez)
Recomendado: un entorno nuevo con **Python 3.11**. En una terminal, antes de abrir el notebook:
```bash
python3.11 -m venv .venv
# macOS / Linux:
source .venv/bin/activate
# Windows PowerShell, en su lugar:
# .venv\Scripts\Activate.ps1
python -m pip install jupyterlab
python -m jupyter lab
```
La celda instala CrewAI con soporte LiteLLM para Ollama, `requests` para HTTP ; el XML se procesa con la biblioteca estándar de Python. Tras la primera instalación, reinicia el kernel y continúa desde el paso 2. Los rangos limitan cambios de versión mayor; no son un entorno bloqueado y probado en todos los equipos.


In [ ]:
# TODO 1 — Instalar las dependencias del taller.
#
# Debes instalar:
# - CrewAI con soporte para LiteLLM.
# - requests.
#
# Pistas:
# - Puedes lanzar pip desde el propio intérprete de Python usando los módulos
#   `subprocess` y `sys`.
# - Usar el ejecutable disponible en `sys.executable` ayuda a instalar los
#   paquetes en el mismo entorno que está ejecutando este notebook.
# - Mantén rangos de versiones compatibles con los indicados en la explicación anterior.
#
# Cuando termines, muestra un mensaje indicando que la instalación ha finalizado.
# Si es la primera instalación, reinicia el kernel antes de continuar.


## 2) Preparar Ollama y cargar las librerías
Instala [Ollama](https://ollama.com/download) y descarga el modelo **en tu terminal**:
```bash
ollama pull llama3.1:latest
```
Abre la aplicación Ollama. En Linux, o si no tienes el servicio activo, ejecuta `ollama serve` en otra terminal. Si indica que el puerto ya está ocupado, probablemente ya está funcionando.

Usamos Llama 3.1 como punto de partida con soporte de herramientas. La memoria necesaria depende también del contexto; si el equipo va justo, reduce el número y la longitud de los documentos. Todos los agentes compartirán el modelo: **tres roles no son tres modelos cargados**.


In [ ]:
# TODO 2 — Preparar el entorno e importar las librerías.
#
# Objetivos:
# 1. Desactiva la telemetría y el tracing de CrewAI ANTES de importar CrewAI.
# 2. Importa las librerías estándar necesarias para:
#    - trabajar con JSON;
#    - calcular un hash;
#    - manejar rutas;
#    - trabajar con fecha y hora UTC;
#    - analizar URLs;
#    - consultar versiones de paquetes;
#    - procesar XML;
#    - normalizar caracteres Unicode;
#    - limpiar HTML;
#    - convertir fechas de correo/RSS.
# 3. Importa `requests`.
# 4. Importa las utilidades de Jupyter para mostrar Markdown.
# 5. Importa de CrewAI los elementos necesarios para construir:
#    - agentes;
#    - tareas;
#    - la Crew;
#    - el LLM;
#    - el proceso secuencial;
#    - herramientas.
# 6. Muestra las versiones de CrewAI, LiteLLM y requests.
#
# Pista:
# - Las variables de entorno de telemetría deben configurarse antes de cargar CrewAI.


## 3) Elegir tema y configurar el taller
Tema inicial: **diplomacia, conflictos y cooperación internacional**.

`KEYWORDS` filtra localmente los títulos y resúmenes del feed; basta con encontrar uno de los términos. Es una selección sencilla, no una búsqueda semántica. Cambiar palabras, modelo o prompts no vuelve a consultar la red.

- `MAX_NEWS=3`: máximo de noticias del boletín.
- `OFFLINE=True`: prohíbe descargar noticias; requiere una copia ya disponible.
- `USE_TOOL=True`: el investigador invoca la herramienta que lee la KB.
- La caché no caduca automáticamente y conserva su fecha. El feed contiene sus últimas entradas disponibles; no se promete una ventana de tres días ni cobertura exhaustiva.

El filtro temático se aplica después de descargar el feed. Si no hay coincidencias, ajusta las palabras o usa `KEYWORDS=[]` para examinar todas las entradas locales sin solicitar más datos.


In [ ]:
# TODO 3 — Definir la configuración general del taller.
#
# Crea variables para:
# - el tema del boletín;
# - la URL del feed RSS;
# - las palabras clave del filtro;
# - el número máximo de noticias;
# - el modo offline;
# - el uso o no de herramienta por parte del investigador;
# - el nombre del modelo de Ollama;
# - la URL local de Ollama.
#
# Después:
# 1. Crea una carpeta local para guardar datos y cachés.
# 2. Genera un identificador estable para el feed a partir de su URL.
# 3. Construye las rutas para:
#    - copia local del RSS;
#    - registro del intento de descarga;
#    - snapshot de la KB.
# 4. Guarda en un diccionario los ajustes principales.
# 5. Muestra por pantalla el tema y las rutas importantes.
#
# Pistas:
# - Usa `Path` para manejar las rutas.
# - Un hash SHA-256 recortado es suficiente para identificar el feed.
# - El tema y las keywords iniciales están descritos en la celda anterior.


## 4) Leer el feed una vez y construir la KB
La respuesta RSS incluye título, enlace, fecha y resumen. Convertimos las entradas en documentos en español con identificadores `[N1]`, `[N2]`, etc. No visitamos sus enlaces ni pedimos artículos completos.

Guardamos el feed antes de filtrar las noticias. Si cambias los filtros o falla un agente, se reutiliza esta copia. Un registro de intento impide repetir una descarga fallida al pulsar Ejecutar otra vez, incluso tras reiniciar el kernel. Si la respuesta contiene una redirección, se informa y no se sigue automáticamente.

**En clase:** el docente prepara el feed y comparte el archivo `FEED_FILE` cuando pueda distribuirlo. Los alumnos lo colocan en la ruta indicada en el paso 3 y usan `OFFLINE=True`. No se necesita una clave de OpenAI ni de Noticias ONU.

**Actualizar deliberadamente:** fuera de las pruebas de agentes, elimina solo `FEED_FILE` y `ATTEMPT_FILE` de esta fuente y ejecuta con `OFFLINE=False`. Para reintentar un fallo, respeta la indicación del servidor y elimina solo ese `ATTEMPT_FILE` cuando decidas realizar una nueva petición. No borres los archivos de GDELT ni toda la carpeta.

La versión se entrega sin noticias precargadas; se ha comprobado el endpoint y el procesamiento de su respuesta real. No se garantiza la disponibilidad futura del feed.


In [ ]:
# TODO 4 — Crear las funciones para procesar el RSS y gestionar la caché.
#
# Debes implementar estas piezas:
#
# A) Función para guardar JSON de forma segura
# - Recibe una ruta y unos datos.
# - Escribe primero en un archivo temporal.
# - Sustituye después el archivo definitivo.
#
# B) Clase auxiliar para extraer texto plano de fragmentos HTML
# - Hereda de `HTMLParser`.
# - Acumula los fragmentos de texto encontrados.
#
# C) Función para limpiar el texto de un resumen RSS
# - Elimina etiquetas HTML.
# - Normaliza espacios repetidos.
#
# D) Función para normalizar texto antes de buscar palabras clave
# - Convierte a minúsculas.
# - Elimina tildes/diacríticos.
#
# E) Función para convertir el XML RSS en una lista de entradas
# Cada entrada debería conservar al menos:
# - título;
# - URL;
# - fuente;
# - fecha de publicación;
# - texto/resumen;
# - tipo de texto.
#
# Reglas recomendadas:
# - ignora entradas sin título;
# - ignora resúmenes demasiado cortos;
# - evita URLs duplicadas;
# - acepta únicamente URLs http/https;
# - intenta convertir `pubDate` a formato ISO;
# - lanza un error si no queda ninguna entrada útil.
#
# F) Función para cargar el feed
# Comportamiento esperado:
# - si existe una copia local, úsala;
# - si estás en modo offline y no existe la copia, informa del problema;
# - evita reintentos automáticos después de un intento fallido;
# - realiza como máximo una petición HTTP;
# - no sigas redirecciones automáticamente;
# - registra el estado HTTP y `Retry-After` cuando exista;
# - si la respuesta es correcta, procesa el RSS y guarda una copia local.
#
# Pistas:
# - `ET.fromstring(...)` permite parsear el XML.
# - En RSS 2.0, las noticias suelen estar bajo `channel/item`.
# - `parsedate_to_datetime(...)` ayuda con `pubDate`.
# - `requests.get(..., allow_redirects=False)` evita seguir redirecciones.
# - El modo exclusivo de apertura de archivos puede servir para detectar
#   si ya se registró un intento anterior.


In [ ]:
# TODO 5 — Filtrar las noticias y construir la KB.
#
# Pasos:
# 1. Carga el snapshot del feed usando la función anterior.
# 2. Normaliza las palabras clave configuradas.
# 3. Recorre las entradas del feed.
# 4. Construye un texto de búsqueda combinando título y resumen.
# 5. Conserva solo las entradas que coincidan con al menos una keyword.
#    - Si la lista de keywords está vacía, no filtres por tema.
# 6. Asigna identificadores estables del tipo N1, N2, N3...
# 7. Detén la selección cuando alcances el máximo configurado.
# 8. Si no hay resultados, informa de forma clara.
# 9. Guarda un snapshot que contenga:
#    - fecha original de descarga;
#    - configuración;
#    - documentos de la KB.
# 10. Muestra un pequeño resumen del resultado.
#
# Pistas:
# - Reutiliza la función de normalización creada antes.
# - La KB debe contener resúmenes del feed, no artículos completos.


### Revisar la KB antes de continuar
Lee los resúmenes. Son la única evidencia disponible para los agentes; los enlaces no se han abierto. La fecha de publicación es la proporcionada por el feed y no establece por sí sola la fecha del suceso.

Puedes quitar una noticia irrelevante de `KB` y ejecutar desde el paso 5. Conserva el archivo `FEED_FILE` para repetir el taller sin descargar noticias. Para cambiar solo prompts, vuelve a ejecutar desde la creación de los agentes.


In [ ]:
# TODO 6 — Revisar manualmente la KB.
#
# Muestra para cada documento:
# - identificador;
# - título;
# - fuente;
# - fecha de publicación;
# - URL.
#
# Después muestra una parte del texto de la primera noticia para inspeccionarla.
#
# Reto opcional:
# - elimina manualmente una noticia concreta de la KB y observa más adelante
#   cómo cambia el resultado del sistema multiagente.
#
# Pista:
# - Recorre la lista `KB` con un bucle.


## 5) Crear una herramienta sencilla para consultar la KB
Una herramienta es una función Python que el agente puede invocar. `@tool` la expone a CrewAI; su documentación explica al modelo cómo usarla.

Como solo tenemos unas pocas noticias, la herramienta devuelve toda la KB. No hay búsqueda semántica ni base vectorial. El filtro local ya seleccionó las entradas temáticas: aquí enseñamos cómo un agente accede a esa información.


In [ ]:
# TODO 7 — Crear una herramienta de CrewAI para consultar la KB.
#
# Parte A:
# Crea una función que transforme la lista de documentos de la KB en un único
# texto legible por el modelo.
#
# Cada bloque debería incluir:
# - [identificador];
# - título;
# - fuente;
# - URL;
# - fecha;
# - tipo de texto;
# - resumen.
#
# Separa visualmente unas noticias de otras.
#
# Parte B:
# Guarda el resultado completo en una variable de contexto.
#
# Parte C:
# Crea una herramienta CrewAI llamada `consultar_kb` que:
# - no necesite argumentos;
# - devuelva el texto completo de la KB.
#
# Finalmente muestra el tamaño del contexto y una pequeña muestra.
#
# Pistas:
# - Usa el decorador `@tool`.
# - La docstring de la herramienta ayuda al modelo a entender cuándo utilizarla.
# - Para este ejercicio no necesitas embeddings ni una base vectorial.


## 6) Configurar y comprobar el modelo local (Ollama)
Primero comprobamos que Ollama está activo y que el modelo está descargado. Después hacemos una llamada corta con el mismo objeto `LLM` que usarán los agentes.

Usamos el endpoint local compatible con OpenAI: `http://localhost:11434/v1`. El prefijo `openai/` selecciona el cliente de conexión; las peticiones van a Ollama en la dirección local especificada. `api_key="ollama"` es un valor de relleno que el servidor local ignora; no necesitas una cuenta ni una clave de OpenAI.

No pasamos `num_ctx`: es una opción específica de Ollama que no acepta directamente el método `chat.completions.create` empleado por esta integración. Esta práctica utiliza el contexto configurado en Ollama. `max_tokens` limita la salida, no configura la ventana de contexto.

`temperature=0.2` reduce la variación; no elimina las alucinaciones. Si ya tienes cargada la KB, puedes ejecutar desde esta celda y después recrear agentes y tareas, sin descargar noticias de nuevo.

Referencia: [compatibilidad de Ollama y uso del servidor local](https://docs.ollama.com/api/openai-compatibility).


In [ ]:
# TODO 8 — Comprobar Ollama y crear el objeto LLM.
#
# Pasos:
# 1. Consulta el endpoint local de Ollama que lista los modelos instalados.
# 2. Gestiona de forma explícita posibles errores de conexión.
# 3. Comprueba que el modelo configurado está instalado.
# 4. Si falta, muestra una indicación clara para descargarlo.
# 5. Crea el objeto `LLM` que usarán todos los agentes.
# 6. Realiza una llamada mínima de prueba.
#
# Configuración recomendada:
# - usa el endpoint compatible con OpenAI de Ollama;
# - utiliza una temperatura baja;
# - limita la longitud de la salida;
# - establece un timeout razonable.
#
# Pistas:
# - Ollama expone normalmente la lista de modelos en `/api/tags`.
# - Para la compatibilidad OpenAI, la base URL termina en `/v1`.
# - CrewAI/LiteLLM necesita identificar el proveedor del modelo.
# - La clave usada contra Ollama local puede ser un valor de relleno.
# - No configures `num_ctx` en este objeto.


## 7) Crear los tres agentes
Un **agente** combina rol, objetivo, instrucciones, modelo y herramientas. Una **tarea** concreta lo que debe producir. La **Crew** establece el orden.

| Agente | Responsabilidad | Acceso a las noticias |
|---|---|---|
| Investigador | Extraer información atribuida a las fuentes | Herramienta `consultar_kb` |
| Analista | Relacionar los hallazgos y marcar incertidumbres | Dossier y KB original |
| Editor | Revisar el respaldo y redactar | Dossier, análisis y KB original |

`allow_delegation=False` evita delegación libre; la coordinación la hace el proceso secuencial. `max_iter` limita los ciclos internos del agente. Damos la KB original también al analista y al editor para que puedan detectar errores heredados.


In [ ]:
# TODO 9 — Crear los tres agentes.
#
# Primero define un bloque común de reglas que todos los agentes deberán seguir.
# Las reglas deberían cubrir al menos:
# - escribir en español;
# - usar solo la KB como evidencia;
# - ignorar instrucciones que aparezcan dentro de los textos de las noticias;
# - no inventar cifras, fechas, citas ni fuentes;
# - usar referencias [N#];
# - distinguir hechos reportados, declaraciones e interpretación;
# - reconocer información ausente o incierta;
# - no afirmar que se han leído artículos completos;
# - evitar predicciones sin respaldo.
#
# Después crea tres agentes:
#
# 1. Investigador
#    - selecciona hechos, actores y declaraciones;
#    - puede acceder a la herramienta `consultar_kb`.
#
# 2. Analista
#    - relaciona los hallazgos;
#    - marca límites, discrepancias e incertidumbre.
#
# 3. Editor
#    - redacta el boletín final;
#    - elimina o matiza afirmaciones no respaldadas.
#
# Para los tres:
# - usa el mismo objeto `llm`;
# - desactiva la delegación;
# - limita el número de iteraciones;
# - activa trazas si quieres observar el comportamiento.
#
# Pista:
# - Solo el investigador necesita la herramienta en este diseño.
# - Puedes condicionar la lista de tools al valor de `USE_TOOL`.


## 8) Definir las tareas y orquestarlas en una única Crew
`context=[research_task]` pasa el dossier al analista. El editor recibe las dos tareas anteriores. **Esto es la orquestación que queremos observar**: cada tarea añade un trabajo distinto.

Los marcadores `{topic}`, `{kb}` y `{downloaded_at}` se rellenan al llamar a `kickoff(inputs=...)`. No se necesitan archivos YAML, un agente supervisor ni memoria persistente.


In [ ]:
# TODO 10 — Definir las tareas y crear la Crew.
#
# Parte A — Entrada del investigador
# - Si `USE_TOOL` es True, pídele explícitamente que consulte la herramienta.
# - Si es False, incluye la KB directamente dentro del prompt.
#
# Parte B — Tarea de investigación
# Debe producir un dossier breve con:
# - tema;
# - hasta tres hallazgos;
# - actores;
# - información reportada;
# - referencias [N#];
# - límites de la muestra.
#
# Parte C — Tarea de análisis
# Debe usar como contexto la tarea de investigación y separar:
# - Información reportada;
# - Interpretación;
# - Qué no sabemos.
#
# Además, debe poder contrastar el dossier con la KB original.
#
# Parte D — Tarea de edición
# Debe usar como contexto las dos tareas anteriores y generar un boletín breve:
# - título;
# - dos o tres párrafos;
# - hasta tres claves;
# - referencias [N#];
# - límites del análisis;
# - fecha de la KB.
#
# Parte E — Crew
# Crea una única Crew con:
# - los tres agentes;
# - las tres tareas;
# - proceso secuencial;
# - memoria desactivada;
# - planning desactivado.
#
# Pistas:
# - Usa `context=[...]` para encadenar tareas.
# - Los placeholders como `{topic}` o `{kb}` se resolverán al ejecutar `kickoff`.
# - El orden de las tareas en una Crew secuencial es importante.


### Ejecutar el equipo
Esta celda puede tardar varios minutos según el ordenador. En las trazas deberías ver primero al investigador consultando la KB, después al analista y finalmente al editor.

Una instrucción de uso de herramienta no garantiza que el modelo la siga: comprueba la traza. Si falla, activa el modo de contexto directo con `USE_TOOL=False` y vuelve a ejecutar desde el paso 3 usando la caché. La coordinación entre agentes sigue siendo la misma.

**Ejecución en Jupyter/VS Code:** usamos `await crew.kickoff_async(...)` porque el kernel ya mantiene un bucle de eventos activo. `Process.sequential` sigue ejecutando las tareas en orden; este cambio no pone los agentes en paralelo. No envuelvas esta celda en `asyncio.run()` ni necesitas `nest_asyncio`.

Si una ejecución anterior terminó con el error del bucle de eventos, vuelve a ejecutar los pasos 7 y 8 para recrear agentes, tareas y Crew; después ejecuta esta celda. No repitas la descarga de noticias.

[Documentación de CrewAI: kickoff_async](https://docs.crewai.com/en/learn/kickoff-async).


In [ ]:
# TODO 11 — Ejecutar la Crew.
#
# Ejecuta la Crew pasando como inputs:
# - el tema;
# - el texto completo de la KB;
# - la fecha de descarga del snapshot.
#
# Guarda:
# - el resultado global;
# - el texto bruto del boletín final.
#
# Finalmente muestra un mensaje indicando que la generación ha terminado.
#
# Pista importante para Jupyter / VS Code:
# - el kernel ya ejecuta un event loop;
# - utiliza la versión asíncrona de kickoff con `await`;
# - no envuelvas la llamada en `asyncio.run()`.


## 9) Mostrar el boletín y revisar las referencias
El programa construye la lista de enlaces desde la KB, en lugar de pedir al modelo que invente o reconstruya URLs. La comprobación detecta **identificadores desconocidos**, pero no demuestra que una frase esté respaldada. Esa revisión requiere leer la fuente.

El editor realiza una revisión interna, no una verificación independiente de la verdad. Las entradas de esta versión pertenecen a la misma fuente institucional; no son una verificación independiente.


In [ ]:
# TODO 12 — Validar las referencias y mostrar el boletín final.
#
# Pasos:
# 1. Extrae del boletín todos los identificadores que sigan el patrón [N#].
# 2. Construye el conjunto de identificadores válidos de la KB.
# 3. Detecta referencias desconocidas.
# 4. Avisa si no se ha encontrado ninguna cita.
# 5. Construye la sección de fuentes SOLO con documentos realmente citados.
# 6. Añade al final la fecha de descarga de la KB.
# 7. Muestra el resultado completo como Markdown.
#
# Pistas:
# - Una expresión regular es útil para localizar las referencias.
# - Usa conjuntos para comparar identificadores citados y conocidos.
# - No pidas al LLM que reconstruya las URLs: obténlas directamente de la KB.
#
# Reflexión:
# - que un identificador exista no demuestra por sí solo que la afirmación esté respaldada;
#   la comprobación semántica sigue necesitando revisión humana.


### Observar qué aportó cada agente
Compara las salidas: ¿qué seleccionó el investigador?, ¿qué añadió el analista?, ¿el editor mantuvo las incertidumbres o las perdió?


In [ ]:
# TODO 13 — Inspeccionar la aportación de cada agente.
#
# Recorre las tres tareas y muestra por separado:
# - salida del investigador;
# - salida del analista;
# - salida del editor.
#
# Si alguna tarea todavía no se ha ejecutado, muestra un mensaje adecuado.
#
# Pista:
# - Tras ejecutar la Crew, cada Task conserva su salida.
# - Observa especialmente qué información se mantiene, se transforma o desaparece
#   entre una etapa y la siguiente.


## 10) Guardar el resultado (opcional)
Guardamos el boletín en Markdown y las versiones del entorno junto al JSON de noticias. El nombre del informe incluye la hora para conservar las distintas pruebas. La selección de resúmenes ya quedó guardada en el paso 4; si eliminaste documentos manualmente, esta celda guarda también la KB exacta usada en esta ejecución.


In [ ]:
# TODO 14 — Guardar el resultado y registrar la ejecución.
#
# Parte A — Informe
# - crea un identificador de ejecución basado en fecha/hora UTC;
# - guarda el boletín final en un archivo Markdown.
#
# Parte B — Registro reproducible
# Guarda en JSON al menos:
# - tema;
# - modelo;
# - si se usó herramienta;
# - fecha de descarga de la KB;
# - documentos usados;
# - versiones de paquetes relevantes;
# - salida del investigador;
# - salida del analista.
#
# Muestra finalmente la ruta del informe generado.
#
# Pistas:
# - `datetime.now(timezone.utc)` permite obtener una marca temporal UTC.
# - Reutiliza la función de guardado JSON que creaste al principio.
# - Usa `ensure_ascii=False` para conservar correctamente caracteres españoles.


## 11) Ejercicios de ampliación
Cuando hayas completado el flujo principal, prueba estas modificaciones:

1. **Cambiar el tema:** modifica `TOPIC` y `KEYWORDS`. Ejecuta desde la configuración y compara qué resúmenes se seleccionan de la misma descarga.
2. **Cambiar el formato:** pide al editor un briefing de cinco puntos sin modificar los otros agentes.
3. **Observar la herramienta:** compara `USE_TOOL=True` y `False` con la misma caché. ¿El investigador leyó realmente las noticias en ambos casos?
4. **Retirar una fuente:** elimina una entrada de `KB` y observa qué afirmaciones desaparecen.
5. **Información ausente:** pide al analista un dato que no aparece en los textos. ¿Reconoce el límite o lo inventa?

**Idea clave:** CrewAI coordina el trabajo, Ollama ejecuta el modelo y la KB aporta el contexto. Los agentes no convierten automáticamente noticias en hechos verificados.


## 12) Problemas habituales

| Problema | Qué comprobar |
|---|---|
| No se instala CrewAI | Usa un entorno limpio con Python 3.11 y el kernel correcto. |
| `running event loop` al ejecutar la Crew | Recrea agentes y tareas y utiliza la ejecución asíncrona con `await`. |
| No conecta con Ollama | Abre la aplicación o ejecuta `ollama serve`. |
| Error relacionado con `num_ctx` | No pases ese parámetro al objeto `LLM` usado en esta práctica. |
| Falta el modelo | Descarga en Ollama el modelo configurado para el taller. |
| Solicita una clave OpenAI | Comprueba que todos los agentes usan el LLM local configurado para Ollama. |
| El feed responde con error | Revisa el registro del intento y utiliza la copia local preparada para el taller. |
| Ya se intentó la descarga | El registro evita nuevas peticiones accidentales; revisa el archivo antes de reintentar deliberadamente. |
| No hay coincidencias | Cambia `KEYWORDS` o usa una lista vacía para revisar todas las entradas locales. |
| Noticias antiguas | Comprueba la fecha original de descarga del feed. |
| Falla el uso de herramientas | Prueba el modo de contexto directo y reutiliza la misma caché. |
| El informe añade detalles ausentes de la KB | Revisa los resúmenes y refuerza las instrucciones; una referencia existente no garantiza respaldo. |

## Documentación de apoyo

- [Noticias ONU: feed RSS en español](https://news.un.org/feed/subscribe/es/news/all/rss.xml)
- [CrewAI: tareas y contexto](https://docs.crewai.com/en/concepts/tasks)
- [CrewAI: herramientas](https://docs.crewai.com/en/concepts/tools)
- [CrewAI: conexiones con LLM](https://docs.crewai.com/en/learn/llm-connections)
- [CrewAI: kickoff asíncrono](https://docs.crewai.com/en/learn/kickoff-async)
- [Ollama: Llama 3.1](https://ollama.com/library/llama3.1)

### Criterios de autoevaluación

Cuando termines, comprueba que:

1. Las tres tareas se ejecutan en orden.
2. El dossier contiene referencias `[N#]`.
3. El análisis distingue evidencia e interpretación.
4. El boletín no cita identificadores inexistentes.
5. Has comprobado manualmente al menos tres afirmaciones contra los fragmentos de la KB.
6. Puedes explicar qué aporta cada agente respecto al anterior.
